<a href="https://colab.research.google.com/github/askarbekkk/kyrgyz-embeddings/blob/main/kyrgyz-embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets -q
from datasets import load_dataset
import random

ds = load_dataset("Zhantas/Cleaned-Kyrgyz_Wikipedia", split="train")

chunks = []

for art in ds.select(range(5000)):
  for para in art["text"].split("\n"):
    if 200 < len(para) < 800:
      chunks.append({"title": art["title"], "text": para.strip()})

print(len(chunks))
random.seed(42)

for c in random.sample(chunks, 3):
  print(c["title"], "|", c["text"][:200], "\n")



README.md:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 67.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/76519 [00:00<?, ? examples/s]

14250
Саймалыташ сүрөт галереясы | Саймалыташ сүрөт галереясын 1902-жылы орус армиясынын офицери, топограф Н.Г.Хлудов ачкан. Кийин аны генерал –мойор И.Т. Пославский (1902-1903), Б.М.Зима (1946), А.Н.Бернштам(1950) изилдеген. Саймалыта 

Кыргыз тилинин грамматикасы | Баяндагыч ыңгай: Кыймыл-аракеттин ошол учурда болуп өткөнүн, болуп жатканын, боло турганын жайынча баяндаган этиш сөздөр баяндагыч ыңгай деп аталат. Мисалы: Шаарга бардым. Шаарга бара жатам. Шаарга ба 

Айыл чарба | Асыл тукум мал чарбасын 228 чарба жүргүзүүчү субъект түзөт. Айыл чарбаны пландоо жана кеңеш берүү жактан камсыз кылуу үчүн агрардык илим жана консулътациялык кызмат борбору, анын карамагындагы төрт ил 



In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')


In [16]:

!pip install google-genai -q

from google import genai
from google.colab import userdata
import random, time, json

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

PROMPT = """You are given a passage in the Kyrgyz language.

Write ONE short question in Kyrgyz that can be answered using this passage.
The question must sound like something a real person would type into a search engine — natural, specific, and self-contained.

Rules:
- Write the question in Kyrgyz only.
- Do not copy full sentences from the passage.
- Do not reference "the text" or "the passage".
- Return only the question, with no explanation or extra formatting.

Passage:
{text}"""

random.seed(42)
sample = random.sample(chunks, 300)
MODELS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-3.6-flash"]

pairs = []
for i, c in enumerate(sample):
    for attempt in range(3):
        try:
            r = client.models.generate_content(
                model=MODELS[attempt % len(MODELS)],
                contents=PROMPT.format(text=c["text"]),
            )
            print("####")
            pairs.append({
                "query": r.text.strip(),
                "positive": c["text"],
                "title": c["title"],
            })
            break
        except Exception as e:
            print(f"[{i}] attempt {attempt+1}: {e}")
            time.sleep(10 * (attempt + 1))

    if i % 25 == 0:
        print(f"processed {i}, collected {len(pairs)}")
    time.sleep(4.2)

print(f"\nDONE: {len(pairs)} pairs")

import json
with open("pairs_raw.json", "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)

for p in pairs[:5]:
    print(p["query"], "\n→", p["positive"][:150], "\n")

from google.colab import drive
drive.mount('/content/drive')

import json
with open("/content/drive/MyDrive/pairs_raw.json", "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)
print("saved", len(pairs))

####
processed 0, collected 1
[1] attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
####
####
####
####
####
[6] attempt 1: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}
####
####
####
####
####
####
[12] attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
####
[13] attempt 1: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}
####
####
####
[16] attempt 1: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
####
####
[18] attempt 1: 

In [13]:
c = sample[0]
r = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=PROMPT.format(text=c["text"]),
)
print("QUERY:", r.text.strip())
print("\nTEXT:", c["text"][:300])

QUERY: Саймалыташ галереясын ким ачкан?

TEXT: Саймалыташ сүрөт галереясын 1902-жылы орус армиясынын офицери, топограф Н.Г.Хлудов ачкан. Кийин аны генерал –мойор И.Т. Пославский (1902-1903), Б.М.Зима (1946), А.Н.Бернштам(1950) изилдеген. Саймалыташтагы эстеликтерде артына куйрук сымал нерсеси бар адамдардын сөлөкөттөрү сакталган. Алардын көпчүлү


In [17]:
from google.colab import drive
drive.mount('/content/drive')

import json
with open("/content/drive/MyDrive/pairs_raw.json", "w", encoding="utf-8") as f:
    json.dump(pairs, f, ensure_ascii=False, indent=2)
print("saved", len(pairs))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved 298
